# Modelo de Machine Learning - Predição Semanal de Demanda

Este notebook treina um classificador para prever a probabilidade da demanda SEMANAL ser Baixa, Média ou Alta, com seleção automática de alvo e detecção robusta de feriados.

In [1]:
# Instalação de biblioteca de feriados (se necessário)
try:
    import holidays
except ImportError:
    !pip install holidays
    import holidays

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. CARGA E SELEÇÃO AUTOMÁTICA
df = pd.read_csv('data/database_final.csv', low_memory=False)
df['DATA_ATEND'] = pd.to_datetime(df['DATA_ATEND'])

# Seleciona automaticamente o par Categoria/Filial com maior volume
top_alvo = df.groupby(['CATEGORIA', 'FILIAL']).size().reset_index(name='CONTAGEM')
top_alvo = top_alvo.sort_values(by='CONTAGEM', ascending=False).iloc[0]

cat_alvo = top_alvo['CATEGORIA']
filial_alvo = top_alvo['FILIAL']

print(f"Treinando para: {cat_alvo} na {filial_alvo}")

df_filtrado = df[(df['CATEGORIA'] == cat_alvo) & (df['FILIAL'] == filial_alvo)].copy()
df_semanal = df_filtrado.set_index('DATA_ATEND').resample('W')['FATUR_VENDA'].sum().reset_index()
df_semanal.columns = ['DATA_INICIO_SEMANA', 'DEMANDA']

Treinando para: Temperos & Condimentos na SHOPPING


## 2. Rotulação e Engenharia de Atributos Robustas

In [2]:
# Limites Semanais (Quantis)
q33 = df_semanal['DEMANDA'].quantile(0.33)
q66 = df_semanal['DEMANDA'].quantile(0.66)

def rotular_demanda(valor):
    if valor <= q33: return 'Baixa'
    if valor <= q66: return 'Média'
    return 'Alta'

df_semanal['CLASSE'] = df_semanal['DEMANDA'].apply(rotular_demanda)

# Detecção Dinâmica de Feriados
br_holidays = holidays.Brazil()

def tem_feriado_robusto(row):
    data_fim = row['DATA_INICIO_SEMANA']
    periodo = pd.date_range(start=data_fim - pd.Timedelta(days=6), end=data_fim)
    for data in periodo:
        if data in br_holidays: return 1
    return 0

df_semanal['MES'] = df_semanal['DATA_INICIO_SEMANA'].dt.month
df_semanal['TEM_FERIADO'] = df_semanal.apply(tem_feriado_robusto, axis=1)

X = df_semanal[['MES', 'TEM_FERIADO']]
y = df_semanal['CLASSE']

## 3. Treinamento e Interface de Predição

In [3]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

def prever_probabilidades_robusto(ano, mes, dia):
    # Identifica se a semana daquela data específica tem feriado
    data_alvo = pd.to_datetime(f"{ano}-{mes:02d}-{dia:02d}")
    
    # Uma semana (7 dias) que contém essa data
    # Para simplificar, verificamos os 3 dias antes e 3 dias depois dessa data
    periodo = pd.date_range(start=data_alvo - pd.Timedelta(days=3), end=data_alvo + pd.Timedelta(days=3))
    feriado_flag = 0
    for data in periodo:
        if data in br_holidays: 
            feriado_flag = 1
            break
    
    input_data = pd.DataFrame({'MES': [mes], 'TEM_FERIADO': [feriado_flag]})
    probs = model.predict_proba(input_data)[0]
    
    print(f"\nPredição Semanal para data em {data_alvo.date()}:")
    print(f"- Feriado Detectado na Semana: {'Sim' if feriado_flag else 'Não'}")
    for i, classe in enumerate(model.classes_):
        print(f"- {classe}: {probs[i]*100:.2f}%")
    
    return dict(zip(model.classes_, probs))

# Exemplo: Semana do 7 de Setembro de 2025
prever_probabilidades_robusto(2025, 9, 7)


Predição Semanal para data em 2025-09-07:
- Feriado Detectado na Semana: Sim
- Alta: 94.58%
- Baixa: 4.42%
- Média: 1.00%


{'Alta': np.float64(0.9458333333333333),
 'Baixa': np.float64(0.04416666666666666),
 'Média': np.float64(0.01)}